# [5.5] Toy Discrete Diffusion Language Models and Local DiffusionGemma Proof - Exercises

In autoregressive generation, the model commits to token 0, then token 1, then token 2. A discrete diffusion language model starts from a corrupted sequence and repeatedly fills in uncertain positions. This notebook builds that loop in a small setting before inspecting the GPU verification report for a trained tiny denoiser and the pinned DiffusionGemma NVFP4 proof.

The key claim boundary is narrow: you will implement toy diffusion mechanics, see a tiny CUDA model-organism pass controls, and inspect one real released-checkpoint generation proof. This is not yet a released DiffusionGemma mechanistic-interpretability notebook.

In [ ]:
import json
import sys
from dataclasses import dataclass
from pathlib import Path

import torch as t
import torch.nn as nn
import torch.nn.functional as F

chapter = "chapter5_modern_architectures"
section = "part5_diffusion_language_models"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part5_diffusion_language_models.tests as tests

MAIN = __name__ == "__main__"


@dataclass(frozen=True)
class DiscreteDiffusionSchedule:
    mask_probs: t.Tensor
    mask_token_id: int

    @property
    def num_steps(self) -> int:
        return int(self.mask_probs.numel())


@dataclass(frozen=True)
class NoisingResult:
    noisy_tokens: t.Tensor
    mask: t.Tensor
    timesteps: t.Tensor


@dataclass(frozen=True)
class DenoisingStepStats:
    step: int
    mask_fraction: float
    mean_entropy: float
    committed_fraction: float

## 1. Mask Schedule

The forward process chooses how much of the sequence to hide. We store schedules from low noise to high noise, because training examples are created by selecting a noising timestep. Sampling will later run this ladder backward.

<details><summary>Expected output</summary>

```text
All tests in `test_linear_mask_schedule_and_expected_fraction` passed!
```

</details>

<details><summary>Help - why low-to-high?</summary>

Forward noising asks how corrupted an example should be. Sampling starts at the other end: fully masked, then less masked, then clean. Keeping the schedule low-to-high makes timestep 0 mean "almost clean" everywhere in the notebook.

</details>

In [ ]:
def linear_mask_schedule(
    num_steps: int,
    *,
    mask_token_id: int,
    min_mask_prob: float = 0.0,
    max_mask_prob: float = 1.0,
) -> DiscreteDiffusionSchedule:
    raise NotImplementedError()


def expected_mask_fraction(schedule: DiscreteDiffusionSchedule, timesteps: t.Tensor) -> float:
    raise NotImplementedError()


tests.test_linear_mask_schedule_and_expected_fraction(
    linear_mask_schedule,
    expected_mask_fraction,
)

<details><summary>Solution</summary>

```python
def linear_mask_schedule(num_steps, *, mask_token_id, min_mask_prob=0.0, max_mask_prob=1.0):
    if num_steps <= 0:
        raise ValueError("num_steps must be positive.")
    if not 0 <= min_mask_prob <= max_mask_prob <= 1:
        raise ValueError("mask probabilities must satisfy 0 <= min <= max <= 1.")
    mask_probs = t.linspace(min_mask_prob, max_mask_prob, num_steps)
    return DiscreteDiffusionSchedule(mask_probs=mask_probs, mask_token_id=mask_token_id)


def expected_mask_fraction(schedule, timesteps):
    probs = schedule.mask_probs.to(device=timesteps.device, dtype=t.float32)[timesteps]
    return probs.mean().item()
```

</details>

## 2. Forward Noising

At timestep 0, no tokens should be masked. At the highest timestep, every token should be masked. In between, noising is independently sampled per token, not per sequence.

<details><summary>Expected output</summary>

```text
All tests in `test_forward_noising_extremes_and_seeded_masks` passed!
```

</details>

<details><summary>Help - per-token noising</summary>

If you flip one coin for the entire sequence, the model sees whole examples disappear at once. A diffusion denoiser needs mixed patterns where some positions are visible context and others are reconstruction targets.

</details>

In [ ]:
def apply_forward_noising(
    input_ids: t.Tensor,
    timesteps: t.Tensor,
    schedule: DiscreteDiffusionSchedule,
    *,
    generator: t.Generator | None = None,
) -> NoisingResult:
    raise NotImplementedError()


tests.test_forward_noising_extremes_and_seeded_masks(
    apply_forward_noising,
    linear_mask_schedule,
)

<details><summary>Solution</summary>

```python
def apply_forward_noising(input_ids, timesteps, schedule, *, generator=None):
    if input_ids.ndim != 2:
        raise ValueError("input_ids must have shape (batch, seq).")
    if timesteps.shape != (input_ids.shape[0],):
        raise ValueError("timesteps must have shape (batch,).")
    if timesteps.min() < 0 or timesteps.max() >= schedule.num_steps:
        raise ValueError("timesteps are out of range for schedule.")
    probs = schedule.mask_probs.to(device=input_ids.device, dtype=t.float32)[timesteps]
    random_values = t.rand(input_ids.shape, generator=generator, device=input_ids.device)
    mask = random_values < probs[:, None]
    noisy = input_ids.clone()
    noisy[mask] = schedule.mask_token_id
    return NoisingResult(noisy_tokens=noisy, mask=mask, timesteps=timesteps)
```

</details>

## 3. Masked Denoising Loss

The model predicts every original token, but the objective should only score positions that were actually hidden by the forward process. Visible tokens are context, not targets.

<details><summary>Expected output</summary>

```text
All tests in `test_masked_denoising_loss_uses_only_masked_positions` passed!
```

</details>

<details><summary>Help - unmasked tokens are context</summary>

A loss over all tokens rewards copying visible inputs. That can make metrics look good while the model fails at the actual diffusion task: reconstructing missing positions.

</details>

In [ ]:
def masked_denoising_loss(logits: t.Tensor, target_ids: t.Tensor, mask: t.Tensor) -> t.Tensor:
    raise NotImplementedError()


tests.test_masked_denoising_loss_uses_only_masked_positions(masked_denoising_loss)

<details><summary>Solution</summary>

```python
def masked_denoising_loss(logits, target_ids, mask):
    if logits.shape[:-1] != target_ids.shape or target_ids.shape != mask.shape:
        raise ValueError("logits, target_ids, and mask shapes are incompatible.")
    if not mask.any():
        raise ValueError("masked_denoising_loss requires at least one masked token.")
    return F.cross_entropy(logits[mask.bool()].float(), target_ids[mask.bool()].long())
```

</details>

## 4. Remasking

After each denoising pass, the sampler fills every position with its current prediction, then masks the least confident positions for the next pass. Uniform remasking is a control: same mask budget, no confidence signal.

<details><summary>Expected output</summary>

```text
All tests in `test_confidence_remask_entropy_and_uniform_control` passed!
```

</details>

<details><summary>Help - confidence is the mechanism under test</summary>

A strong sampler should commit to easy positions earlier. Uniform remasking lets you ask whether the confidence heuristic matters, or whether any remasking pattern with the same budget would work.

</details>

In [ ]:
def token_entropy(logits: t.Tensor) -> t.Tensor:
    raise NotImplementedError()


def confidence_remask(
    logits: t.Tensor,
    current_tokens: t.Tensor,
    *,
    mask_token_id: int,
    next_mask_fraction: float,
) -> t.Tensor:
    raise NotImplementedError()


def uniform_remask(
    tokens: t.Tensor,
    *,
    mask_token_id: int,
    next_mask_fraction: float,
    generator: t.Generator | None = None,
) -> t.Tensor:
    raise NotImplementedError()


tests.test_confidence_remask_entropy_and_uniform_control(
    confidence_remask,
    token_entropy,
    uniform_remask,
)

<details><summary>Solution</summary>

```python
def token_entropy(logits):
    log_probs = F.log_softmax(logits.float(), dim=-1)
    probs = log_probs.exp()
    return -(probs * log_probs).sum(dim=-1)


def confidence_remask(logits, current_tokens, *, mask_token_id, next_mask_fraction):
    if not 0 <= next_mask_fraction <= 1:
        raise ValueError("next_mask_fraction must be in [0, 1].")
    probs = F.softmax(logits.float(), dim=-1)
    confidence, predictions = probs.max(dim=-1)
    new_tokens = predictions.to(dtype=current_tokens.dtype)
    num_to_mask = int(round(next_mask_fraction * current_tokens.shape[1]))
    if num_to_mask == 0:
        return new_tokens
    low_conf = confidence.topk(k=num_to_mask, dim=-1, largest=False).indices
    new_tokens.scatter_(1, low_conf, mask_token_id)
    return new_tokens


def uniform_remask(tokens, *, mask_token_id, next_mask_fraction, generator=None):
    if not 0 <= next_mask_fraction <= 1:
        raise ValueError("next_mask_fraction must be in [0, 1].")
    num_to_mask = int(round(next_mask_fraction * tokens.shape[1]))
    if num_to_mask == 0:
        return tokens.clone()
    scores = t.rand(tokens.shape, generator=generator, device=tokens.device)
    chosen = scores.topk(k=num_to_mask, dim=-1, largest=False).indices
    remasked = tokens.clone()
    remasked.scatter_(1, chosen, mask_token_id)
    return remasked
```

</details>

## 5. Oracle Sampler

Before training a model, make the sampler recover a known target from an oracle denoiser. This isolates sampler bugs from model-learning failures.

<details><summary>Expected output</summary>

```text
All tests in `test_oracle_diffusion_sampler_recovers_target` passed!
```

</details>

<details><summary>Help - reverse the schedule</summary>

Sampling starts at the highest noise level. Each loop predicts a full sequence and decides which positions remain masked for the next, lower-noise step.

</details>

In [ ]:
def diffusion_sampler(
    model_fn,
    *,
    shape: tuple[int, int],
    schedule: DiscreteDiffusionSchedule,
    temperature: float = 0.0,
    remask: str = "confidence",
    generator: t.Generator | None = None,
    device: t.device | None = None,
) -> tuple[t.Tensor, list[DenoisingStepStats]]:
    raise NotImplementedError()


tests.test_oracle_diffusion_sampler_recovers_target(
    diffusion_sampler,
    linear_mask_schedule,
)

<details><summary>Solution</summary>

```python
def diffusion_sampler(model_fn, *, shape, schedule, temperature=0.0, remask="confidence", generator=None, device=None):
    if device is None:
        device = schedule.mask_probs.device
    tokens = t.full(shape, schedule.mask_token_id, dtype=t.long, device=device)
    stats = []
    for step in reversed(range(schedule.num_steps)):
        logits = model_fn(tokens, step)
        if temperature == 0.0:
            predictions = logits.argmax(dim=-1)
        else:
            probs = F.softmax(logits.float() / temperature, dim=-1)
            samples = t.multinomial(probs.reshape(-1, probs.shape[-1]), 1, generator=generator)
            predictions = samples.reshape(tokens.shape)
        if step == 0:
            tokens = predictions.to(dtype=tokens.dtype)
        else:
            next_fraction = float(schedule.mask_probs[step - 1].item())
            if remask == "confidence":
                tokens = confidence_remask(logits, predictions, mask_token_id=schedule.mask_token_id, next_mask_fraction=next_fraction)
            elif remask == "uniform":
                tokens = uniform_remask(predictions, mask_token_id=schedule.mask_token_id, next_mask_fraction=next_fraction, generator=generator)
            else:
                raise ValueError("remask must be 'confidence' or 'uniform'.")
        mask_fraction = tokens.eq(schedule.mask_token_id).float().mean().item()
        stats.append(DenoisingStepStats(step=step, mask_fraction=mask_fraction, mean_entropy=token_entropy(logits).mean().item(), committed_fraction=1.0 - mask_fraction))
    return tokens, stats
```

</details>

## 6. Denoising-Step Diagnostics

A diffusion trajectory gives you more than a final sample. Commitment time asks when each token first became unmasked, and activation trajectory validation is the minimum shape contract before future patching work.

<details><summary>Expected output</summary>

```text
All tests in `test_commitment_edit_distance_and_activation_trajectory` passed!
```

</details>

<details><summary>Help - trajectory index vs diffusion timestep</summary>

The sampler stores rows while walking backward through the schedule. Commitment time is the first row where a token is visible, not the numerical diffusion timestep printed inside that row.

</details>

In [ ]:
def commitment_times(tokens_over_steps: t.Tensor, mask_token_id: int) -> t.Tensor:
    raise NotImplementedError()


def edit_distance(a: list[int], b: list[int]) -> int:
    raise NotImplementedError()


def validate_activation_trajectory(
    activations: list[t.Tensor],
    *,
    expected_steps: int,
    batch: int,
    seq_len: int,
) -> bool:
    raise NotImplementedError()


tests.test_commitment_edit_distance_and_activation_trajectory(
    commitment_times,
    edit_distance,
    validate_activation_trajectory,
)

<details><summary>Solution</summary>

```python
def commitment_times(tokens_over_steps, mask_token_id):
    if tokens_over_steps.ndim != 3:
        raise ValueError("tokens_over_steps must have shape (steps, batch, seq).")
    unmasked = tokens_over_steps.ne(mask_token_id)
    any_unmasked = unmasked.any(dim=0)
    first = unmasked.float().argmax(dim=0).long()
    return t.where(any_unmasked, first, t.full_like(first, -1))


def edit_distance(a, b):
    prev = list(range(len(b) + 1))
    for i, token_a in enumerate(a, start=1):
        cur = [i]
        for j, token_b in enumerate(b, start=1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + int(token_a != token_b)))
        prev = cur
    return prev[-1]


def validate_activation_trajectory(activations, *, expected_steps, batch, seq_len):
    if len(activations) != expected_steps:
        return False
    return all(act.shape[0] == batch and act.shape[1] == seq_len for act in activations)
```

</details>

## 7. Tiny Bidirectional Denoiser

The CUDA report trains this architecture on the generated copy-pair grammar `[a, b] -> [a, b, a, a, b, b]`. Here you only check the shape contract: token, position, and timestep embeddings should produce token-level logits.

<details><summary>Expected output</summary>

```text
All tests in `test_tiny_conditional_diffusion_lm_forward_shape` passed!
```

</details>

<details><summary>Help - no causal mask</summary>

A diffusion denoiser predicts masked positions using the full corrupted sequence. The information hiding comes from mask tokens, not from causal attention.

</details>

In [ ]:
class TinyConditionalDiffusionLM(nn.Module):
    def __init__(
        self,
        *,
        vocab_size: int = 11,
        seq_len: int = 6,
        num_steps: int = 6,
        d_model: int = 96,
    ) -> None:
        super().__init__()
        self.seq_len = seq_len
        self.token_embed = nn.Embedding(vocab_size, d_model)
        self.position_embed = nn.Embedding(seq_len, d_model)
        self.time_embed = nn.Embedding(num_steps, d_model)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=4,
            dim_feedforward=2 * d_model,
            batch_first=True,
            activation="gelu",
        )
        self.transformer = nn.TransformerEncoder(layer, num_layers=2)
        self.unembed = nn.Linear(d_model, vocab_size)

    def forward(self, input_ids: t.Tensor, timesteps: t.Tensor) -> t.Tensor:
        raise NotImplementedError()


tests.test_tiny_conditional_diffusion_lm_forward_shape(TinyConditionalDiffusionLM)

<details><summary>Solution</summary>

```python
class TinyConditionalDiffusionLM(nn.Module):
    ...
    def forward(self, input_ids: t.Tensor, timesteps: t.Tensor) -> t.Tensor:
        positions = t.arange(self.seq_len, device=input_ids.device)
        hidden = (
            self.token_embed(input_ids)
            + self.position_embed(positions)[None, :, :]
            + self.time_embed(timesteps)[:, None, :]
        )
        return self.unembed(self.transformer(hidden))
```

</details>

## Whole-Notebook Contract

Once your implementations pass the local cells, compare them against the section contract. This is intentionally still a smoke test: the full CUDA run is represented by the committed verification report below.

<details><summary>Expected output</summary>

```text
All tests in `test_notebook_contract` passed!
```

</details>

In [ ]:
# Uncomment after completing the exercises above.
# from part5_diffusion_language_models.solutions import run_smoke_test
# tests.test_notebook_contract(run_smoke_test)

## Signature Result

The current report contains two separate results: a tiny CUDA-trained diffusion model-organism, and a scoped released-checkpoint DiffusionGemma NVFP4 generation proof. The table below reads the committed report so the notebook displays the same evidence as CI.

<details><summary>Interpreting the signature result</summary>

Held-out masked accuracy and sampler exact match prove the tiny denoiser learned the generated grammar under the declared controls. The shuffled-label accuracy shows the result is not explained by a trivial prior over suffix tokens. The NVFP4 proof shows that the released checkpoint generated locally through an isolated vLLM runtime; it is not denoising-time circuit evidence.

</details>

In [ ]:
def _load_committed_gpu_report() -> dict:
    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    assert gpu["cuda_available"]
    assert gpu["within_vram_budget"]
    return gpu


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


gpu = run_gpu_test()
{
    "heldout_masked_accuracy": gpu["heldout_masked_accuracy"],
    "sampler_exact_match": gpu["sampler_exact_match"],
    "shuffled_label_accuracy": gpu["shuffled_label_accuracy"],
    "entropy_by_step": [round(x, 4) for x in gpu["entropy_by_step"]],
    "diffusiongemma_generation_ready": gpu["diffusiongemma_generation_ready"],
    "nvfp4_isolated_vllm_generation_ready": gpu["diffusiongemma_nvfp4_isolated_vllm_generation_ready"],
    "peak_vram_gb": round(gpu["peak_vram_gb"], 3),
}

## Limitations

This section is a mechanics lab plus a released-checkpoint feasibility proof. It does not claim DiffusionGemma denoising-step activation capture, diffusion-time patching, throughput benchmarking, broad quality evaluation, or BF16 direct inference on this 24GB machine. Those become future exercises only after the runtime exposes the needed internals.